# Preprocesamiento de datos — Capa Bronze

Análisis exploratorio sobre los datos crudos ingestados en `nessie.nyc` para
justificar las reglas de limpieza aplicadas en silver. Se cubren cuatro bloques por
tabla: tipos de datos, datos faltantes, outliers y valores inválidos. Una última
sección resume cada regla silver con la evidencia cuantitativa encontrada en bronze.

Tablas analizadas: `nessie.nyc.yellow_taxi` y `nessie.nyc.taxi_zone_lookup`.

In [1]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie

spark = get_spark("preprocess-bronze")
nessie.set_spark_branch(spark, "bronze")
spark.version

26/05/31 16:45:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.8'

## 1. Yellow Taxi

In [2]:
yt = spark.table("nessie.nyc.yellow_taxi").cache()
total_yt = yt.count()
print(f"Total de registros: {total_yt:,}")
print(f"Columnas: {len(yt.columns)}")

Total de registros: 30,137,291
Columnas: 20


### 1.1 Análisis de tipos de datos

La ingesta bronze unifica el *schema drift* del origen NYC TLC: entre años los
parquet originales alternan `INT`/`BIGINT` en los IDs, `INT`/`DOUBLE` en
`passenger_count` y `RatecodeID`, y renombran `Airport_fee` → `airport_fee` en 2025.
Bronze los normaliza a un esquema canónico con *casts* explícitos; aquí verificamos
el resultado.

In [3]:
# Esquema canónico resultante tras la ingesta
yt.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [4]:
# Histórico de evolución del esquema (Iceberg)
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-05-31 16:38:57.482|751264134898552722 |NULL               |true               |
|2026-05-31 16:39:15.652|179461454785393106 |751264134898552722 |true               |
|2026-05-31 16:39:34.254|8663488671414409298|179461454785393106 |true               |
|2026-05-31 16:39:47.106|3042831213633643212|8663488671414409298|true               |
|2026-05-31 16:40:01.572|8983329294462776866|3042831213633643212|true               |
|2026-05-31 16:40:16.811|1001402227516272329|8983329294462776866|true               |
|2026-05-31 16:40:31.102|3423163687281835035|1001402227516272329|true               |
|2026-05-31 16:40:45.461|6547510629800845180|3423163687281835035|true               |
|2026-05-31 16:41:03.05 |1757265477960251573|654751062

In [5]:
# cbd_congestion_fee (añadida en 2025) solo debería tener valores desde ese año.
# Lo validamos contando no-nulos por año: el grueso se concentra en 2025, como
# se espera. Los pocos no-nulos en años anteriores (p.ej. 21 en 2024) son las
# filas con timestamps fuera de rango (ver 1.3.1) — basura del taxímetro que la
# ventana de carga Q1/años objetivo descarta de todos modos.
from pyspark.sql.functions import col, year as spark_year, sum as spark_sum, when

yt.groupBy(spark_year(col("tpep_pickup_datetime")).alias("year")) \
  .agg(
      spark_sum(when(col("cbd_congestion_fee").isNotNull(), 1).otherwise(0)).alias("cbd_no_null"),
      spark_sum(when(col("airport_fee").isNotNull(), 1).otherwise(0)).alias("airport_fee_no_null"),
      spark_sum(when(col("congestion_surcharge").isNotNull(), 1).otherwise(0)).alias("congestion_no_null"),
  ) \
  .orderBy("year") \
  .show(50)

+----+-----------+-------------------+------------------+
|year|cbd_no_null|airport_fee_no_null|congestion_no_null|
+----+-----------+-------------------+------------------+
|2001|          0|                  3|                 3|
|2002|          0|                  6|                 6|
|2003|          0|                  3|                 3|
|2007|          1|                  1|                 1|
|2008|          0|                  9|                 9|
|2009|          1|                  6|                 6|
|2014|          0|                  1|                 1|
|2022|          0|                 36|                36|
|2023|          0|            9148264|           9148264|
|2024|         21|            8802818|           8802818|
|2025|   11198003|            8934254|           8934254|
+----+-----------+-------------------+------------------+



### 1.2 Análisis de datos faltantes

Distinguimos dos tipos de nulos:
- **Estructurales**: columnas que no existían en determinados años (p.ej.
  `congestion_surcharge` desde 2019, `airport_fee` poblada efectivamente
  desde 2022 en este dataset, `cbd_congestion_fee` desde 2025). No son
  errores y no se filtran en silver.
- **Anómalos**: nulos en columnas clave (`tpep_pickup_datetime`,
  `tpep_dropoff_datetime`) que impiden cualquier análisis temporal. Silver los
  elimina.

In [6]:
# Nulos totales por columna
null_counts = yt.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in yt.columns
])
null_counts.show(vertical=True)

-RECORD 0-------------------------
 VendorID              | 0        
 tpep_pickup_datetime  | 0        
 tpep_dropoff_datetime | 0        
 passenger_count       | 3251890  
 trip_distance         | 0        
 RatecodeID            | 3251890  
 store_and_fwd_flag    | 3251890  
 PULocationID          | 0        
 DOLocationID          | 0        
 payment_type          | 0        
 fare_amount           | 0        
 extra                 | 0        
 mta_tax               | 0        
 tip_amount            | 0        
 tolls_amount          | 0        
 improvement_surcharge | 0        
 total_amount          | 0        
 congestion_surcharge  | 3251890  
 airport_fee           | 3251890  
 cbd_congestion_fee    | 18939265 



In [7]:
# Porcentaje de nulos (sólo columnas con al menos un nulo)
row = null_counts.collect()[0].asDict()
pct = [(c, n, round(100 * n / total_yt, 2)) for c, n in row.items() if n > 0]
pct.sort(key=lambda t: -t[1])
for c, n, p in pct:
    print(f"  {c:<25s}  {n:>10,}  ({p:5.2f}%)")

  cbd_congestion_fee         18,939,265  (62.84%)
  passenger_count             3,251,890  (10.79%)
  RatecodeID                  3,251,890  (10.79%)
  store_and_fwd_flag          3,251,890  (10.79%)
  congestion_surcharge        3,251,890  (10.79%)
  airport_fee                 3,251,890  (10.79%)


In [8]:
# Nulos por año en columnas con nulos estructurales vs anómalos.
# Confirma que congestion_surcharge y cbd_congestion_fee son estructurales
# (nulos concentrados en años anteriores a su introducción).
nulls_year = yt.groupBy(spark_year(col("tpep_pickup_datetime")).alias("year")).agg(
    spark_sum(col("passenger_count").isNull().cast("int")).alias("passenger_count_null"),
    spark_sum(col("RatecodeID").isNull().cast("int")).alias("RatecodeID_null"),
    spark_sum(col("store_and_fwd_flag").isNull().cast("int")).alias("store_and_fwd_null"),
    spark_sum(col("congestion_surcharge").isNull().cast("int")).alias("congestion_null"),
    spark_sum(col("airport_fee").isNull().cast("int")).alias("airport_fee_null"),
    spark_sum(col("cbd_congestion_fee").isNull().cast("int")).alias("cbd_null"),
).orderBy("year")
nulls_year.show(50)

+----+--------------------+---------------+------------------+---------------+----------------+--------+
|year|passenger_count_null|RatecodeID_null|store_and_fwd_null|congestion_null|airport_fee_null|cbd_null|
+----+--------------------+---------------+------------------+---------------+----------------+--------+
|2001|                   0|              0|                 0|              0|               0|       3|
|2002|                   0|              0|                 0|              0|               0|       6|
|2003|                   0|              0|                 0|              0|               0|       3|
|2007|                   0|              0|                 0|              0|               0|       0|
|2008|                   0|              0|                 0|              0|               0|       9|
|2009|                   0|              0|                 0|              0|               0|       5|
|2014|                   0|              0|            

In [9]:
# Filas con timestamps ausentes — no recuperables.
# Reutilizamos null_counts ya calculado arriba (evita dos escaneos completos).
print(f"tpep_pickup_datetime NULL:  {row['tpep_pickup_datetime']:,}")
print(f"tpep_dropoff_datetime NULL: {row['tpep_dropoff_datetime']:,}")

tpep_pickup_datetime NULL:  0
tpep_dropoff_datetime NULL: 0


### 1.3 Análisis de outliers y valores inválidos

Cada apartado muestra la evidencia detrás de una regla de limpieza silver.

#### 1.3.1 Timestamps fuera de rango

El dataset contiene viajes con fechas imposibles (años 2001, 2008, 2014, 2022…),
errores del taxímetro. Silver solo ingesta el primer trimestre de los años
objetivo (2023, 2024, 2025): el bucle WAP de `clean_yellow_taxi.py`
filtra cada mes por `year`/`month`, de modo que estos timestamps fuera de rango
quedan excluidos por la propia ventana de carga, sin necesidad de una regla en
`clean()`.

In [10]:
# Distribución de años fuera del rango esperado (2023-2025 de las cargas Q1)
from pyspark.sql.functions import month as spark_month

year_counts = yt.groupBy(
    spark_year(col("tpep_pickup_datetime")).alias("year")
).count().orderBy("year")
year_counts.show(50)

+----+--------+
|year|   count|
+----+--------+
|2001|       3|
|2002|       6|
|2003|       3|
|2007|       1|
|2008|       9|
|2009|       6|
|2014|       1|
|2022|      36|
|2023| 9384443|
|2024| 9554780|
|2025|11198003|
+----+--------+



#### 1.3.2 Timestamps inconsistentes (pickup ≥ dropoff)

Viajes cuyo *dropoff* es anterior o igual al *pickup*: imposibles físicamente.

In [11]:
bad_ts = yt.filter(col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime")).count()
print(f"Filas con pickup >= dropoff: {bad_ts:,}  ({100 * bad_ts / total_yt:.3f}%)")

Filas con pickup >= dropoff: 36,135  (0.120%)


#### 1.3.3 Importes no positivos y distancias negativas

Valores negativos en `trip_distance`, `fare_amount`, `tip_amount`,
`tolls_amount` y `total_amount` no tienen interpretación física en un viaje
normal (en el feed TLC suelen ser ajustes contables y *chargebacks*).

Además, un viaje real siempre tiene tarifa positiva: `fare_amount = 0` o
`total_amount = 0` señala un registro erróneo (viaje anulado, fallo del
taxímetro), no un trayecto válido. Por eso silver endurece esas dos reglas de
`>= 0` a `> 0`, mientras que `trip_distance`, `tip_amount` y `tolls_amount`
conservan el `0` como valor legítimo (un viaje puede no tener propina ni peajes).

In [12]:
# Una sola pasada con agg() en vez de 5 filter().count() independientes.
neg_checks = [
    ("trip_distance", col("trip_distance") < 0),
    ("fare_amount",   col("fare_amount") < 0),
    ("total_amount",  col("total_amount") < 0),
    ("tip_amount",    col("tip_amount") < 0),
    ("tolls_amount",  col("tolls_amount") < 0),
]
neg_counts = yt.agg(*[
    spark_sum(when(cond, 1).otherwise(0)).alias(name)
    for name, cond in neg_checks
]).collect()[0].asDict()
for name, _ in neg_checks:
    n = neg_counts[name]
    print(f"  {name:<15s}  negativos: {n:>8,}  ({100 * n / total_yt:.3f}%)")

  trip_distance    negativos:        0  (0.000%)
  fare_amount      negativos:  751,554  (2.494%)
  total_amount     negativos:  382,664  (1.270%)
  tip_amount       negativos:    1,421  (0.005%)
  tolls_amount     negativos:   26,555  (0.088%)


In [13]:
# Importes exactamente a 0: un viaje real siempre tiene tarifa positiva, así que
# fare_amount == 0 o total_amount == 0 es un registro erróneo. Silver endurece
# estas dos reglas a "> 0" (trip_distance, tip_amount y tolls_amount mantienen
# el 0 como valor legítimo y siguen con ">= 0").
# Una sola pasada con agg() en vez de dos filter().count() independientes.
zeros = yt.agg(
    spark_sum(when(col("fare_amount") == 0, 1).otherwise(0)).alias("zero_fare"),
    spark_sum(when(col("total_amount") == 0, 1).otherwise(0)).alias("zero_total"),
).collect()[0]
zero_fare, zero_total = zeros["zero_fare"], zeros["zero_total"]
print(f"fare_amount == 0:  {zero_fare:>9,}  ({100 * zero_fare / total_yt:.3f}%)")
print(f"total_amount == 0: {zero_total:>9,}  ({100 * zero_total / total_yt:.3f}%)")

fare_amount == 0:     10,905  (0.036%)
total_amount == 0:     4,628  (0.015%)


In [14]:
# Resumen estadístico: min/max muestran la magnitud del ruido
yt.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "tolls_amount"
).summary("min", "25%", "50%", "75%", "max").show()

+-------+-------------+-----------+----------+------------+------------+
|summary|trip_distance|fare_amount|tip_amount|total_amount|tolls_amount|
+-------+-------------+-----------+----------+------------+------------+
|    min|          0.0|    -1807.6|    -300.0|    -1832.85|     -142.17|
|    25%|         1.01|        8.6|       0.0|       15.48|         0.0|
|    50%|         1.73|       12.8|      2.58|       20.27|         0.0|
|    75%|         3.25|       20.5|      4.06|        28.8|         0.0|
|    max|    335004.33|  863372.12|    999.99|   863380.37|      916.87|
+-------+-------------+-----------+----------+------------+------------+



#### 1.3.4 Outliers en importes, distancia y duración

Importes y distancias correctamente firmados pero con valores extremos. Los
percentiles altos (p95–p99.9) frente al máximo miden la longitud de la cola
derecha, y los Top 20 por `fare_amount` y `trip_distance` revelan su
naturaleza: junto a viajes caros plausibles (negociados con `RatecodeID = 5`,
trayectos al aeropuerto) aparecen valores imposibles — tarifas de cientos de
miles de dólares con `trip_distance = 0`, o distancias de cientos de miles de
millas en trayectos de pocos minutos.

Esa cola derecha distorsiona el entrenamiento de los modelos ML: las métricas
de error cuadrático (RMSE, R²) quedan dominadas por un puñado de filas
extremas y dejan de ser informativas. Por ello silver aplica, tras el filtrado
de validez, una **eliminación de outliers por rango intercuartílico (IQR) por
`RatecodeID`**: cada código tarifario (metro estándar 1, JFK plano 2, Newark
plano 3, Nassau/Westchester 4, negociado 5, grupo 6, desconocido 99) es una
población de tarifas distinta; una sola valla global queda dominada por el
código 1 (~99,5 % de las filas) y recortaría viajes legítimos al aeropuerto o
negociados. Se descartan las filas fuera de las vallas de Tukey
`[Q1 − k·IQR, Q3 + k·IQR]` con `k = 3.0` (ajuste conservador *far out*) sobre
`fare_amount`, `total_amount`, `trip_distance` y la duración del viaje, en
ambas colas. La última celda calcula esas vallas por código sobre la
rebanada bronze ya validada — réplica directa de `remove_outliers()` en
`clean_yellow_taxi.py` — y cuantifica cuántas filas caen fuera.

In [15]:
# Percentiles altos: revelan la forma de la cola derecha. Si p99 y max están
# muy separados hay una cola larga de valores atípicos candidata a ser recortada.
yt.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "tolls_amount"
).summary("50%", "90%", "95%", "99%", "99.9%", "max").show()

+-------+-------------+-----------+----------+------------+------------+
|summary|trip_distance|fare_amount|tip_amount|total_amount|tolls_amount|
+-------+-------------+-----------+----------+------------+------------+
|    50%|         1.73|       12.8|      2.58|       20.27|         0.0|
|    90%|         8.22|       37.3|       6.6|       50.67|         0.0|
|    95%|        12.65|       57.6|     10.79|       77.39|        6.55|
|    99%|        19.74|       73.0|     17.19|      102.75|        6.94|
|  99.9%|         28.7|      133.2|      27.4|      163.74|       20.82|
|    max|    335004.33|  863372.12|    999.99|   863380.37|      916.87|
+-------+-------------+-----------+----------+------------+------------+



In [16]:
# Top 20 por fare_amount — ¿son negociados (RatecodeID=5), viajes al
# aeropuerto o basura? Revisar visualmente decide si el tope superior
# captura ruido o recorta viajes legítimos caros.
yt.select(
    "tpep_pickup_datetime", "trip_distance", "fare_amount", "total_amount",
    "RatecodeID", "payment_type", "PULocationID", "DOLocationID"
).orderBy(col("fare_amount").desc()).limit(20).show(truncate=False)

+--------------------+-------------+-----------+------------+----------+------------+------------+------------+
|tpep_pickup_datetime|trip_distance|fare_amount|total_amount|RatecodeID|payment_type|PULocationID|DOLocationID|
+--------------------+-------------+-----------+------------+----------+------------+------------+------------+
|2025-01-20 12:07:18 |1.6          |863372.12  |863380.37   |1.0       |4           |138         |8           |
|2025-02-21 17:28:43 |2.2          |132531.36  |132555.41   |1.0       |3           |161         |246         |
|2025-03-09 00:34:23 |1191.9       |46263.88   |46269.44    |1.0       |2           |229         |262         |
|2024-02-29 07:56:45 |0.0          |9792.0     |9792.0      |NULL      |0           |262         |140         |
|2024-01-20 11:18:47 |0.0          |5000.0     |5000.0      |99.0      |3           |264         |264         |
|2024-01-20 11:19:33 |0.0          |5000.0     |5000.0      |99.0      |3           |264         |264   

In [17]:
# Top 20 por trip_distance — un viaje dentro de NYC rara vez supera ~40 mi
# (JFK → norte de Westchester). Valores de cientos de millas son errores del
# taxímetro; deberían caer fuera de cualquier cota superior razonable.
yt.select(
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
    "fare_amount", "RatecodeID", "PULocationID", "DOLocationID"
).orderBy(col("trip_distance").desc()).limit(20).show(truncate=False)

+--------------------+---------------------+-------------+-----------+----------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|RatecodeID|PULocationID|DOLocationID|
+--------------------+---------------------+-------------+-----------+----------+------------+------------+
|2023-02-21 05:19:00 |2023-02-21 05:36:00  |335004.33    |17.75      |NULL      |142         |42          |
|2025-03-27 20:52:00 |2025-03-27 20:59:00  |320136.29    |-4.75      |NULL      |161         |163         |
|2024-01-30 06:37:00 |2024-01-30 06:50:00  |312722.3     |14.46      |NULL      |151         |162         |
|2025-03-27 06:32:00 |2025-03-27 06:42:00  |281085.57    |9.98       |NULL      |215         |130         |
|2025-03-22 05:11:00 |2025-03-22 05:31:00  |280125.56    |21.53      |NULL      |215         |258         |
|2025-01-22 21:04:00 |2025-01-22 21:13:00  |276423.57    |-4.75      |NULL      |161         |170         |
|2025-01-19 14:51:00 |2025-0

In [ ]:
# Vallas de Tukey (k=3) calculadas POR RatecodeID, sobre la rebanada bronze
# ya validada. Réplica de remove_outliers() en clean_yellow_taxi.py:
# un solo groupBy(RatecodeID).agg(count, percentile_approx) -> .collect() ->
# CASE WHEN incrustado que evalúa las cuatro columnas en una sola pasada.
#
# Diferencias deliberadas con el pipeline:
#  - El pipeline opera sobre UN MES validado; aquí cubrimos toda la ventana
#    bronze validada (Q1 de los años objetivo) para dimensionar el descarte
#    a nivel global.
#  - Se omite dropDuplicates() del filtrado de validez por coste (68 M filas;
#    ver 1.3.8: ~13 k duplicados, efecto despreciable en cuartiles).
#  - Se conserva el conteo `n` por código sólo para la tabla informativa;
#    el pipeline no lo usa (todas las vallas se aplican por igual).
# Las cifras resultantes son INDICATIVAS: el pipeline recalcula las vallas
# mes a mes y código a código. `valid`, `per_code_fences`, `out_of_fence_expr`
# y las constantes IQR_K / OUTLIER_COLS / DURATION_COL
# se reutilizan en la sección 3.
from pyspark.sql.functions import (
    count as spark_count, lit, percentile_approx, unix_timestamp,
)

IQR_K = 3.0
QUANTILE_ACCURACY = 1000  # inverso del relative error 0.001 — mismo valor que el pipeline
DURATION_COL = "_trip_duration_s"
TARGET_YEARS = [2023, 2024, 2025]
TARGET_MONTHS = [1, 2, 3]

# Rebanada validada: misma cadena de filtros que clean() (sin dropDuplicates).
# La columna de duración se añade después de los filtros, igual que en el
# pipeline, y se usa solo como entrada para su propia valla.
valid = (
    yt
    .filter(spark_year(col("tpep_pickup_datetime")).isin(TARGET_YEARS))
    .filter(spark_month(col("tpep_pickup_datetime")).isin(TARGET_MONTHS))
    .filter(col("tpep_pickup_datetime").isNotNull())
    .filter(col("tpep_dropoff_datetime").isNotNull())
    .filter(col("tpep_pickup_datetime") < col("tpep_dropoff_datetime"))
    .filter(col("trip_distance") >= 0)
    .filter(col("fare_amount") > 0)
    .filter(col("total_amount") > 0)
    .filter(col("tip_amount") >= 0)
    .filter(col("tolls_amount") >= 0)
    .filter(col("PULocationID").between(1, 265))
    .filter(col("DOLocationID").between(1, 265))
    .filter(col("VendorID").isin([1, 2, 6, 7]))
    .filter(col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99]))
    .filter(col("payment_type").isin([0, 1, 2, 3, 4, 5, 6]))
    .filter(col("store_and_fwd_flag").isin(["Y", "N"]))
    .filter(col("passenger_count").between(1, 6))
    .withColumn(
        DURATION_COL,
        unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime"),
    )
).cache()
valid_count = valid.count()
print(f"Tras filtrado de validez: {valid_count:,} filas ({100 * valid_count / total_yt:.1f} % de bronze)")

OUTLIER_COLS = ["fare_amount", "total_amount", "trip_distance", DURATION_COL]

# Un único groupBy(RatecodeID) materializa Q1/Q3 y n por código.
fence_rows = valid.groupBy("RatecodeID").agg(
    spark_count("*").alias("n"),
    *[percentile_approx(c, [0.25, 0.75], QUANTILE_ACCURACY).alias(c)
      for c in OUTLIER_COLS],
).orderBy("RatecodeID").collect()

# Tabla por columna: una valla por RatecodeID.
for c in OUTLIER_COLS:
    print(f"\n{c}:")
    print(f"  {'RatecodeID':>10s}  {'n':>11s}  {'[Q1 - 3·IQR, Q3 + 3·IQR]':>30s}")
    print(f"  {'-'*10}  {'-'*11}  {'-'*30}")
    for r in fence_rows:
        code = "NULL" if r["RatecodeID"] is None else f"{r['RatecodeID']:.0f}"
        q1, q3 = r[c]
        iqr = q3 - q1
        lo, hi = q1 - IQR_K * iqr, q3 + IQR_K * iqr
        fence_str = f"[{lo:>9.2f}, {hi:>9.2f}]"
        print(f"  {code:>10s}  {r['n']:>11,}  {fence_str:>30s}")

# Vallas por código en estructura accesible para la sección 3.
per_code_fences = {}  # {RatecodeID: {col: (lo, hi)}}
for r in fence_rows:
    per_code_fences[r["RatecodeID"]] = {
        c: (
            r[c][0] - IQR_K * (r[c][1] - r[c][0]),
            r[c][1] + IQR_K * (r[c][1] - r[c][0]),
        )
        for c in OUTLIER_COLS
    }


def out_of_fence_expr(c):
    """CASE WHEN RatecodeID == X THEN NOT col.between(lo, hi) ... ELSE False.

    Inverso del CASE WHEN baked-in de remove_outliers(): cada RatecodeID
    devuelve `~col.between(lo, hi)` con su propia valla; el default `False`
    cubre cualquier código sin valla.
    """
    expr = lit(False)
    for code, fences_c in per_code_fences.items():
        lo, hi = fences_c[c]
        expr = when(col("RatecodeID") == code, ~col(c).between(lo, hi)).otherwise(expr)
    return expr


out_counts = valid.agg(*[
    spark_sum(when(out_of_fence_expr(c), 1).otherwise(0)).alias(c)
    for c in OUTLIER_COLS
]).collect()[0].asDict()

# Mostramos el descarte con DOS denominadores para que sea directamente
# comparable con la sección 3 (que normaliza siempre sobre bronze):
#   - % sobre validadas: lo que ve remove_outliers() en el pipeline.
#   - % sobre bronze:    misma base que el resumen final de reglas.
print(f"\nFilas fuera de la valla   (% sobre {valid_count:,} validadas / % sobre {total_yt:,} bronze):")
for c in OUTLIER_COLS:
    n = out_counts[c]
    pct_valid = 100 * n / valid_count
    pct_bronze = 100 * n / total_yt
    print(f"  {c:<20s} {n:>10,}  ({pct_valid:>5.2f} % validadas / {pct_bronze:>5.2f} % bronze)")

#### 1.3.5 IDs de localización fuera de rango

`PULocationID` y `DOLocationID` deben referenciar una zona válida (1–265 según
la tabla de referencia TLC). Los IDs 264 ("Unknown") y 265 ("N/A") se conservan
intencionadamente en silver — descartamos solo lo que está fuera del rango.

In [19]:
# Una sola pasada con agg() en vez de dos filter().count() independientes.
loc_out = yt.agg(
    spark_sum(when(~col("PULocationID").between(1, 265), 1).otherwise(0)).alias("pu"),
    spark_sum(when(~col("DOLocationID").between(1, 265), 1).otherwise(0)).alias("do"),
).collect()[0]
print(f"PULocationID fuera de [1,265]: {loc_out['pu']:,}")
print(f"DOLocationID fuera de [1,265]: {loc_out['do']:,}")

PULocationID fuera de [1,265]: 0
DOLocationID fuera de [1,265]: 0


#### 1.3.6 Códigos categóricos inválidos

Valores que no figuran en el diccionario de datos TLC 2024 indican
corrupción del taxímetro o registros de prueba.

Se aplica un único diccionario —el de la edición 2024— de forma uniforme a
todos los años ingestados (2023-2025). Por eso `VendorID` 6 y 7 se aceptan
también en años anteriores a su aparición oficial, y 4 y 5 se descartan en
todos: el objetivo es un criterio de validez estable para el lakehouse, no
reconstruir el diccionario vigente año a año.

In [20]:
# Valores presentes vs. valores válidos según diccionario TLC
print("VendorID (válidos 1,2,6,7):")
yt.groupBy("VendorID").count().orderBy("VendorID").show(truncate=False)

print("RatecodeID (válidos 1-6, 99):")
yt.groupBy("RatecodeID").count().orderBy("RatecodeID").show(truncate=False)

print("payment_type (válidos 0-6):")
yt.groupBy("payment_type").count().orderBy("payment_type").show(truncate=False)

print("store_and_fwd_flag (válidos Y, N):")
yt.groupBy("store_and_fwd_flag").count().orderBy("store_and_fwd_flag").show(truncate=False)

VendorID (válidos 1,2,6,7):
+--------+--------+
|VendorID|count   |
+--------+--------+
|1       |7235143 |
|2       |22870039|
|6       |5002    |
|7       |27107   |
+--------+--------+

RatecodeID (válidos 1-6, 99):
+----------+--------+
|RatecodeID|count   |
+----------+--------+
|NULL      |3251890 |
|1.0       |25360456|
|2.0       |938173  |
|3.0       |80925   |
|4.0       |54766   |
|5.0       |184329  |
|6.0       |44      |
|99.0      |266708  |
+----------+--------+

payment_type (válidos 0-6):
+------------+--------+
|payment_type|count   |
+------------+--------+
|0           |3251890 |
|1           |22153789|
|2           |4041562 |
|3           |192219  |
|4           |497829  |
|5           |2       |
+------------+--------+

store_and_fwd_flag (válidos Y, N):
+------------------+--------+
|store_and_fwd_flag|count   |
+------------------+--------+
|NULL              |3251890 |
|N                 |26752313|
|Y                 |133088  |
+------------------+--------+



In [21]:
# Recuento agregado de filas con al menos un código inválido — una sola pasada.
# Aquí NULL no cuenta como código inválido (es dato faltante, ver 1.2): ~isin()
# devuelve NULL para entrada NULL y when() lo trata como falso. El resumen final
# (sección 3) sí los suma, porque mide lo que silver descarta — y filter() los
# elimina. De ahí que RatecodeID/store_and_fwd_flag salgan 0 aquí y >0 allí.
invalid_checks = [
    ("VendorID inválido",            ~col("VendorID").isin([1, 2, 6, 7])),
    ("RatecodeID inválido",          ~col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99])),
    ("payment_type inválido",        ~col("payment_type").isin([0, 1, 2, 3, 4, 5, 6])),
    ("store_and_fwd_flag inválido",  ~col("store_and_fwd_flag").isin(["Y", "N"])),
]
invalid_counts = yt.agg(*[
    spark_sum(when(cond, 1).otherwise(0)).alias(name)
    for name, cond in invalid_checks
]).collect()[0].asDict()
for name, _ in invalid_checks:
    print(f"{name:<32s} {invalid_counts[name]:,}")

VendorID inválido                0
RatecodeID inválido              0
payment_type inválido            0
store_and_fwd_flag inválido      0


#### 1.3.7 Passenger count fuera de lo físicamente posible

La ley NYC TLC permite 5 pasajeros en un taxi estándar, más la excepción de
menores de 7 años en el regazo de un adulto → techo razonable = 6. Los valores
7, 8 y 9 que aparecen en bronze son probables *sentinels* o errores del
taxímetro (no caben físicamente en un sedán de 4–5 plazas) y se descartan.

In [22]:
# Distribución completa (incluye nulos y valores extremos)
yt.groupBy("passenger_count").count().orderBy("passenger_count").show(50, truncate=False)

+---------------+--------+
|passenger_count|count   |
+---------------+--------+
|NULL           |3251890 |
|0.0            |331859  |
|1.0            |20888084|
|2.0            |3829158 |
|3.0            |877792  |
|4.0            |495343  |
|5.0            |280243  |
|6.0            |182721  |
|7.0            |42      |
|8.0            |133     |
|9.0            |26      |
+---------------+--------+



In [23]:
# Filas que silver descarta por la regla passenger_count (incluye 0, 7, 8, 9 y NULL).
# OJO: ~between(1,6) devuelve NULL para passenger_count NULL, y filter()/when()
# tratan NULL como falso -> hay que sumar isNull() explícitamente, igual que el
# filtro de silver, que también elimina los NULL.
# Una sola pasada con agg() en vez de tres filter().count() independientes.
invalid_pc = col("passenger_count").isNull() | ~col("passenger_count").between(1, 6)
pc = yt.agg(
    spark_sum(when(invalid_pc, 1).otherwise(0)).alias("out_range"),
    spark_sum(when(col("passenger_count").isNull(), 1).otherwise(0)).alias("nulls"),
    spark_sum(when(col("passenger_count") == 0, 1).otherwise(0)).alias("zero"),
).collect()[0]
out_range, nulls_pc, zero = pc["out_range"], pc["nulls"], pc["zero"]
print(f"passenger_count fuera de [1,6] o NULL: {out_range:,}  ({100 * out_range / total_yt:.2f}%)")
print(f"  de los cuales NULL:                  {nulls_pc:,}")
print(f"  de los cuales = 0:                   {zero:,}")

passenger_count fuera de [1,6] o NULL: 3,583,950  (11.89%)
  de los cuales NULL:                  3,251,890
  de los cuales = 0:                   331,859


#### 1.3.8 Duplicados exactos

Silver aplica `dropDuplicates()` global como red de seguridad (mismas 20
columnas idénticas). Aquí cuantificamos la magnitud mes a mes: un duplicado
exacto comparte las 20 columnas —`tpep_pickup_datetime` incluido—, así que
las dos copias caen siempre en el mismo mes y nunca cruzan de mes; basta con
deduplicar cada (año, mes) y sumar. Por cada mes evaluamos `distinct()`
(réplica literal de `dropDuplicates()`: compara la fila completa y trata dos
NULL como iguales) sobre ~3–8 M filas, lo que mantiene el *shuffle* dentro de
lo que el clúster soporta sin recurrir a un `dropDuplicates()` global sobre
las 68 M filas de bronze. `countDistinct()` sobre varias columnas no sirve
aquí: por semántica SQL ignora toda fila con algún NULL, y `yellow_taxi`
tiene millones (`passenger_count`, `airport_fee`, etc.).

In [24]:
# Duplicados exactos. Un duplicado exacto comparte las 20 columnas
# —tpep_pickup_datetime incluido—, así que las dos copias caen siempre en el
# mismo mes: nunca cruzan de mes. Por eso basta deduplicar mes a mes y sumar.
# Recorremos los mismos (año, mes) que ingesta el pipeline (Q1 de los años
# objetivo), igual que clean_yellow_taxi.py: la cifra resultante es exactamente
# la que elimina dropDuplicates() en silver.
#
# distinct() es la réplica literal de esa regla: compara la fila completa y
# trata dos NULL como iguales. (countDistinct() sobre varias columnas NO sirve
# aquí: por semántica SQL ignora toda fila con algún NULL, y yellow_taxi tiene
# millones — passenger_count, airport_fee, etc.) Trabajar por mes (~4,5 M filas)
# mantiene el shuffle de distinct() dentro de lo que el clúster soporta.
TARGET_YEARS = [2023, 2024, 2025]
TARGET_MONTHS = [1, 2, 3]

dups_by_month = []
for y in TARGET_YEARS:
    for mth in TARGET_MONTHS:
        month_df = yt.filter(
            (spark_year(col("tpep_pickup_datetime")) == y)
            & (spark_month(col("tpep_pickup_datetime")) == mth)
        )
        rows = month_df.count()
        if rows == 0:
            continue
        distinct_rows = month_df.distinct().count()
        dups_by_month.append((f"{y}-{mth:02d}", rows, rows - distinct_rows))

total_rows = sum(r for _, r, _ in dups_by_month)
dups = sum(d for _, _, d in dups_by_month)

print(f"{'mes':<9s} {'filas':>13s} {'duplicados':>12s}")
print("-" * 36)
for label, rows, d in dups_by_month:
    print(f"{label:<9s} {rows:>13,} {d:>12,}")
print("-" * 36)
print(f"{'TOTAL':<9s} {total_rows:>13,} {dups:>12,}  ({100 * dups / total_rows:.3f}%)")

mes               filas   duplicados
------------------------------------
2023-01       3,066,726            0
2023-02       2,914,003            0
2023-03       3,403,619            0
2024-01       2,964,617            0
2024-02       3,007,533            1
2024-03       3,582,607            0
2025-01       3,475,234            0
2025-02       3,577,542            0
2025-03       4,145,225            0
------------------------------------
TOTAL        30,137,106            1  (0.000%)


## 2. Taxi Zone Lookup

In [25]:
zl = spark.table("nessie.nyc.taxi_zone_lookup").cache()
total_zl = zl.count()
print(f"Total de registros: {total_zl}")
print(f"Columnas: {len(zl.columns)}")

Total de registros: 265
Columnas: 4


### 2.1 Análisis de tipos de datos

La tabla de referencia llega en CSV y sus tipos son estables. Silver renombra
las columnas a *snake_case* (`LocationID` → `location_id`, `Borough` →
`borough`, …) para homogeneizar el estilo del lakehouse.

In [26]:
zl.printSchema()

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



### 2.2 Análisis de datos faltantes

Toda zona debería tener `LocationID`, `Borough` y `Zone`. `service_zone` sí
puede ser nulo en alguna zona administrativa.

In [27]:
# Nulos por columna
null_counts_zl = zl.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in zl.columns
])
null_counts_zl.show()

+----------+-------+----+------------+
|LocationID|Borough|Zone|service_zone|
+----------+-------+----+------------+
|         0|      0|   0|           0|
+----------+-------+----+------------+



In [28]:
# Strings vacíos o solo whitespace (no detectados como nulos pero igual de problemáticos)
from pyspark.sql.functions import trim, length

empty_borough = zl.filter((length(trim(col("Borough"))) == 0) | col("Borough").isNull()).count()
empty_zone = zl.filter((length(trim(col("Zone"))) == 0) | col("Zone").isNull()).count()
empty_service = zl.filter((length(trim(col("service_zone"))) == 0) | col("service_zone").isNull()).count()
print(f"Borough vacío o nulo:      {empty_borough}")
print(f"Zone vacío o nulo:         {empty_zone}")
print(f"service_zone vacío o nulo: {empty_service}")

Borough vacío o nulo:      0
Zone vacío o nulo:         0
service_zone vacío o nulo: 0


### 2.3 Análisis de outliers y valores inválidos

In [29]:
# LocationID fuera del rango esperado [1, 265]
out_range_zl = zl.filter(~col("LocationID").between(1, 265)).count()
print(f"LocationID fuera de [1,265]: {out_range_zl}")

# Duplicados en la clave
dup_ids = zl.groupBy("LocationID").count().filter(col("count") > 1).count()
print(f"LocationID duplicados:       {dup_ids}")

LocationID fuera de [1,265]: 0
LocationID duplicados:       0


In [30]:
# Valores sentinel "Unknown" / "N/A" — se conservan (264, 265 son casos válidos del TLC)
zl.filter(
    col("Borough").isin(["Unknown", "N/A"]) | col("Zone").isin(["Unknown", "N/A"])
).show(truncate=False)

+----------+-------+--------------+------------+
|LocationID|Borough|Zone          |service_zone|
+----------+-------+--------------+------------+
|264       |Unknown|N/A           |N/A         |
|265       |N/A    |Outside of NYC|N/A         |
+----------+-------+--------------+------------+



In [31]:
# Dominio de boroughs — esperamos los cinco distritos más Unknown / N/A
# (sentinel para las zonas 264 y 265) y EWR (Newark Airport, pseudo-distrito
# legítimo del diccionario TLC). Sin valores fuera de este conjunto = no hay typos.
zl.groupBy("Borough").count().orderBy(col("count").desc()).show(truncate=False)

+-------------+-----+
|Borough      |count|
+-------------+-----+
|Queens       |69   |
|Manhattan    |69   |
|Brooklyn     |61   |
|Bronx        |43   |
|Staten Island|20   |
|Unknown      |1    |
|N/A          |1    |
|EWR          |1    |
+-------------+-----+



## 3. Resumen — reglas silver justificadas

Se recogen en una sola tabla las reglas aplicadas en
`src/pipelines/silver/clean_yellow_taxi.py` y `clean_zone_lookup.py`, junto al
recuento de filas bronze que cada regla descarta (evidencia cuantitativa).

Cada regla de validez se expresa como su condición de **conservación**: una
fila se descarta cuando esa condición no es verdadera — falsa **o NULL** —, tal
como hace `filter()` en Spark. Por eso las columnas con nulos estructurales
(`RatecodeID`, `store_and_fwd_flag`, `passenger_count`) suman aquí también esos
NULL, y sus cifras superan el recuento de *códigos inválidos* de 1.3.6 (que solo
mira valores no nulos). La primera fila refleja la ventana de carga del bucle
WAP (primer trimestre de los años objetivo), no una regla de `clean()`.

La fila **duplicados exactos** corresponde al `dropDuplicates()` global de
`clean()` (apartado 1.3.8). No es una condición de conservación por fila, así
que se mide aparte, como diferencia entre filas totales y filas distintas.

Las cuatro últimas filas son la regla de **eliminación de outliers IQR por
`RatecodeID`** (apartado 1.3.4): silver descarta las filas fuera de las vallas
de Tukey `[Q1 − 3·IQR, Q3 + 3·IQR]` calculadas para cada código tarifario. Se
evalúan sobre la rebanada `valid` (ya filtrada por las reglas de validez) y se
reportan como porcentaje de bronze para ser comparables con el resto de la
tabla. Son cifras **indicativas**: el pipeline recalcula las vallas mes a mes
y código a código, mientras que aquí se calculan una sola vez sobre toda la
ventana bronze validada. Sirven para dimensionar el orden de magnitud del
recorte, no para sumarse con las reglas de validez.

In [ ]:
# Cada regla aislada contra bronze — útil para el informe final del TFM.
# Una sola pasada con agg() en vez de un escaneo por regla.
# Cada regla de validez es la condición de CONSERVACIÓN de silver: la fila se
# descarta si no se cumple (falsa o NULL), igual que filter(). Sumar
# keep.isNull() es lo que evita el infraconteo silencioso de las columnas
# con nulos estructurales.
# Nota: las reglas se aplican en cadena en silver; aquí se miden por separado
# (la suma no es la resta total porque una fila puede violar varias).
#
# Reutilizamos lo ya calculado más arriba para no repetir escaneos sobre 68 M
# filas: `valid`, `per_code_fences`, `out_of_fence_expr`, `OUTLIER_COLS` y
# `DURATION_COL` vienen de 1.3.4; `dups` viene de 1.3.8. Las cifras IQR son
# INDICATIVAS — vallas globales sobre la ventana bronze validada, no por mes
# como en el pipeline real.
TARGET_YEARS = [2023, 2024, 2025]
TARGET_MONTHS = [1, 2, 3]

print("vallas IQR por RatecodeID (rango sobre los códigos con valla):")
for c in OUTLIER_COLS:
    los = [f[c][0] for f in per_code_fences.values()]
    his = [f[c][1] for f in per_code_fences.values()]
    print(f"  {c:<20s} lo ∈ [{min(los):>9.2f}, {max(los):>9.2f}]   "
          f"hi ∈ [{min(his):>9.2f}, {max(his):>9.2f}]")

keep_rules = [
    ("pickup en Q1 de años objetivo",
        spark_year(col("tpep_pickup_datetime")).isin(TARGET_YEARS) &
        spark_month(col("tpep_pickup_datetime")).isin(TARGET_MONTHS)),
    ("pickup no NULL",            col("tpep_pickup_datetime").isNotNull()),
    ("dropoff no NULL",           col("tpep_dropoff_datetime").isNotNull()),
    ("pickup < dropoff",          col("tpep_pickup_datetime") < col("tpep_dropoff_datetime")),
    ("trip_distance >= 0",        col("trip_distance") >= 0),
    ("fare_amount > 0",           col("fare_amount") > 0),
    ("total_amount > 0",          col("total_amount") > 0),
    ("tip_amount >= 0",           col("tip_amount") >= 0),
    ("tolls_amount >= 0",         col("tolls_amount") >= 0),
    ("PULocationID en [1,265]",   col("PULocationID").between(1, 265)),
    ("DOLocationID en [1,265]",   col("DOLocationID").between(1, 265)),
    ("VendorID válido",           col("VendorID").isin([1, 2, 6, 7])),
    ("RatecodeID válido",         col("RatecodeID").isin([1, 2, 3, 4, 5, 6, 99])),
    ("payment_type válido",       col("payment_type").isin([0, 1, 2, 3, 4, 5, 6])),
    ("store_and_fwd_flag válido", col("store_and_fwd_flag").isin(["Y", "N"])),
    ("passenger_count en [1,6]",  col("passenger_count").between(1, 6)),
]

# Validez: se evalúa sobre bronze (yt). Una fila se descarta si la condición
# es falsa O NULL — replica el comportamiento de filter() en clean().
drop_counts = yt.agg(*[
    spark_sum(when(~keep | keep.isNull(), 1).otherwise(0)).alias(name)
    for name, keep in keep_rules
]).collect()[0].asDict()

# IQR: se evalúa sobre `valid` (post-validez) con la valla del RatecodeID
# correspondiente, reutilizando out_of_fence_expr() definida en 1.3.4.
iqr_drop_counts = valid.agg(*[
    spark_sum(when(out_of_fence_expr(c), 1).otherwise(0)).alias(c)
    for c in OUTLIER_COLS
]).collect()[0].asDict()
iqr_label = {
    "fare_amount":   "fare_amount fuera de valla por RatecodeID",
    "total_amount":  "total_amount fuera de valla por RatecodeID",
    "trip_distance": "trip_distance fuera de valla por RatecodeID",
    DURATION_COL:    "duración fuera de valla por RatecodeID",
}

# dropDuplicates() de clean(): no es una condición de conservación por fila,
# así que se mide aparte como (filas - filas distintas). `dups` ya viene
# calculado de 1.3.8 — no se reescanea aquí.


def _print_row(name, n):
    # % siempre sobre bronze para que toda la tabla sea comparable.
    print(f"{name:<42s} {n:>12,}  {100 * n / total_yt:>6.2f}%")


print(f"\n{'regla de validez silver':<42s} {'descarta':>12s}  {'%':>7s}")
print("-" * 65)
for name, _ in keep_rules:
    _print_row(name, drop_counts[name])
_print_row("duplicados exactos (dropDuplicates)", dups)

print(f"\n{'outliers IQR (indicativo, ver nota)':<42s} {'descarta':>12s}  {'%':>7s}")
print("-" * 65)
for c in OUTLIER_COLS:
    _print_row(iqr_label[c], iqr_drop_counts[c])

In [33]:
# Liberar caches y cerrar la sesión.
# unpersist(blocking=True) espera a que los executors borren los bloques antes
# de stop(): así no queda ninguna RPC en vuelo al cerrarse la conexión, que es
# lo que provoca el ERROR "TransportResponseHandler ... requests outstanding".
yt.unpersist(blocking=True)
zl.unpersist(blocking=True)
valid.unpersist(blocking=True)
spark.stop()